# TD4 — Mini-MCP: turn your PIM into tools an agent can discover

**~1h · the fourth brick of the PIM assistant you build across TD1–TD5.**

In **TD3** you built a RAG retriever over the catalog. In **TD5** you'll build the **agent** that processes a supplier email end-to-end. But an agent that *acts* on the PIM needs **operations** it can call: search the catalog, fetch a product, read the category tree, read a category's attribute schema, create a product.

You *could* wire each operation into your agent's code by hand. But then it only works for **your** code, in **your** process, in **your** language. A different agent — Claude Desktop, a colleague's Python service, a tool written in Go — can't discover or call any of it. That doesn't scale.

**MCP (the Model Context Protocol)** is the fix: a **standard protocol** for exposing tools to AI models — the "USB-C of AI". You write your operations once, behind an MCP **server**, and *any* MCP-speaking client gets a typed, self-describing **catalog of tools** over the wire — **without importing your code**. That's the whole idea: **standardization + discoverability**.

In this lab you will:

1. feel the pain of an **ad-hoc** Python tool (works, but nobody else can find or call it);
2. turn it into an **MCP tool** by adding a decorator, type hints, and a docstring — *your signature becomes the contract*;
3. have a **client discover and call it over the protocol** — without importing your function;
4. expose your **TD3 RAG** (`search_products`) and the **category schema** as discoverable tools.

**The server you build here is exactly what your TD5 agent will connect to** — there, an agent loop drives these tools; here, you build and inspect the toolbox itself.

**How to work:** run the notebook top to bottom; complete the **4 `# TODO`** cells (register a tool · speak the protocol · `search_products` · `get_category_attributes`); **write your own analysis answer** (you may use Claude Code to help you *think*, but the words must be yours); ask Claude Code first whenever you're stuck. Each `# TODO` ends with a quick **self-check** (an `assert` / smoke-test) so you can confirm your code is right **without any solution**.

> ⚠️ TD4 is **fully local and free** — **no API key needed**. Embeddings run locally (`sentence-transformers`) and the MCP server runs **in-process**; nothing here calls a hosted model. The agent that *uses* this server — and makes the real model calls — is **TD5**. 🚀

## 1. Setup

We rebuild the **same RAG retriever as TD3** — the catalog indexed in ChromaDB with the local MiniLM model — because in §5 we expose that retriever as the MCP tool `search_products`. This is self-contained (a few seconds), so TD4 needs no artifact from TD3. The retrieval itself is *not* the lesson here — **exposing it as a tool is** — so `retrieve()` is given to you, pre-built and identical to TD3.

In [ ]:
import json
import logging
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

# MCP server + in-memory client transport (real list_tools / call_tool, no subprocess).
from mcp.server.fastmcp import FastMCP
from mcp.shared.memory import create_connected_server_and_client_session as connect

# The MCP server logs every request at INFO; quiet it so notebook output stays readable.
logging.getLogger("mcp").setLevel(logging.WARNING)

# Shared PIM catalog + taxonomy — the same dataset used in TD1, TD2, TD3.
df = pd.read_csv("../data/products.csv")
df["doc"] = df["name"] + " — " + df["long_description"]
with open("../data/taxonomy.json") as f:
    taxonomy = json.load(f)

print(f"Loaded {len(df)} products across {df['category'].nunique()} leaf categories.")

In [ ]:
# Rebuild the TD3 RAG index: embed the catalog locally with MiniLM and add the vectors to Chroma
# EXPLICITLY (NOT Chroma's default embedder) -- same one shared vector space as TD1 -> TD3 -> TD4 -> TD5.
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
if "catalog" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("catalog")
collection = chroma_client.create_collection("catalog")

embeddings = embed_model.encode(df["doc"].tolist(), show_progress_bar=False)
collection.add(
    ids=df["sku"].tolist(),
    embeddings=embeddings.tolist(),
    documents=df["doc"].tolist(),
    metadatas=[{"name": r["name"], "category": r["category"], "price": float(r["price"]),
                "short_description": r["short_description"]}
               for _, r in df.iterrows()],
)
print(f"Indexed {collection.count()} products into ChromaDB.")

In [ ]:
# Pre-built RAG retriever -- same as TD3, returning each hit's FULL stored metadata.
# (Not the lesson here; exposing it as a tool is.)
def retrieve(query_text, k=3):
    """Return the k catalog products most similar to `query_text`, each as a dict carrying the
    product's FULL stored metadata (sku + everything saved in the DB)."""
    q_vec = embed_model.encode(query_text).tolist()
    res = collection.query(query_embeddings=[q_vec], n_results=k)
    hits = []
    for sku, meta in zip(res["ids"][0], res["metadatas"][0]):
        hit = {"sku": sku, **meta}
        for field in ("attributes", "extra"):   # stored as JSON strings -> parse back to dicts
            if isinstance(hit.get(field), str):
                hit[field] = json.loads(hit[field])
        hits.append(hit)
    return hits

# Smoke test the retriever before we wrap it as a tool.
for h in retrieve("noise cancelling headphones", k=3):
    print(f"  - {h['name']}  ({h['category']}, €{h['price']:.0f})")

## 2. The problem: an ad-hoc tool

Our agent will need the PIM's **category tree** (top categories → leaf categories). The simplest thing that could work: a plain Python function that reads `taxonomy.json`. Let's write it and call it.

In [ ]:
# A plain Python function -- the ad-hoc approach. It reads the taxonomy and returns top -> [leaves].
def get_category_tree_plain():
    """Return the catalog category tree as {top_category: [leaf, ...]}."""
    return {
        cat["name"]: [sub["name"] for sub in cat["subcategories"]]
        for cat in taxonomy["categories"]
    }

tree = get_category_tree_plain()
print(f"{len(tree)} top categories:")
for top, leaves in tree.items():
    print(f"  {top}: {', '.join(leaves)}")

**Takeaway.** It works — but *only* because **this** notebook imported **this** function into **this** Python process. There is no schema, no description, no contract. A different agent — in another process, another language, or Claude Desktop — has **no way to discover that this function exists**, what arguments it takes, or what it returns. To let *any* agent call it, we need a **standard protocol**. That's MCP.

## 3. Build a minimal MCP server

An MCP **server** publishes your functions as **tools** over the protocol. We use **FastMCP**: you decorate a function with `@mcp_server.tool()`, and the decorator **automatically builds the tool's schema** from your **type hints** (parameter names, types, which are required) and its **description** from your **docstring**.

That is the core idea of this section: **you don't write a schema by hand — your signature and docstring *are* the contract the model reads.** So you write them *for the model*.

In [ ]:
# Pre-built: create the server. Tools get registered onto it below.
mcp_server = FastMCP("pim")
print("MCP server created:", mcp_server.name)

In [ ]:
# TODO 1 -- register your first MCP tool.
# Turn the §2 category-tree function into an MCP tool. Concretely:
#   1. define a function `get_category_tree()` that takes NO arguments and RETURNS a dict
#      {top_category: [leaf, ...]} -- you can just call get_category_tree_plain() inside it;
#   2. decorate it with `@mcp_server.tool()` so it is published over the protocol;
#   3. give it a `-> dict` return type hint and a clear ONE-LINE docstring.
# Why it matters: the decorator reads your signature + docstring to build the tool's schema and
# description AUTOMATICALLY. The docstring is what the model will read to decide whether to call the
# tool -- so write it for the model, and make the return type explicit.

# TODO: define and decorate get_category_tree() here
raise NotImplementedError

# Self-check: the tool must be registered on the server under the name 'get_category_tree'.
_registered = await mcp_server.list_tools()
_names = {t.name for t in _registered}
assert "get_category_tree" in _names, f"tool not registered; found {_names}"
print("TODO 1 self-check passed. Registered tools:", sorted(_names))

**Takeaway.** You didn't write a JSON schema or a tool description — you wrote a normal Python function. The `@mcp_server.tool()` decorator turned your **signature + docstring into the contract** the protocol exposes. Next we'll have a *client* read that contract over the wire.

## 4. A client discovers and calls it — without importing your code

Now the other half: an MCP **client**. In a real deployment the client and server run in separate processes (see the mini-project). In the notebook we use an **in-memory transport** — `connect(mcp_server._mcp_server)` wires a client session straight to the server in-process — so you get **real** `list_tools()` / `call_tool()` protocol calls with no subprocess plumbing to manage. (`ipykernel` allows top-level `await`, so a cell can `await` directly.)

> **Predict (10 seconds — just for you, not graded).** The client below **never imports** `get_category_tree`. What do you think `list_tools()` will hand back — and where will the tool's *description* and *schema* come from? Jot a guess, then run the cell.

In [ ]:
# TODO 2 -- speak the MCP protocol from the client side.
# The `async with connect(...)` scaffold below is given. INSIDE it, you must:
#   1. discover the catalog:  listed = await session.list_tools()
#      -> listed.tools is a list of Tool(name, description, inputSchema) objects;
#   2. call the tool:          result = await session.call_tool("get_category_tree", {})
#      -> the {} is the arguments dict (this tool takes none);
#      -> result.content[0].text is a JSON STRING of the tool's return value -> json.loads() it.
# Then print the discovered tool name + description + inputSchema, and the parsed call result.
# Notice: you are reading the description/schema the SERVER generated from your docstring + type
# hints -- you never imported the function.

async with connect(mcp_server._mcp_server) as session:
    # TODO: list the tools, call get_category_tree, parse + print the result
    raise NotImplementedError

    # Self-check (keep these lines inside the `async with`, after your code defines
    # `listed` and `result`):
    # assert "get_category_tree" in {t.name for t in listed.tools}
    # tree = json.loads(result.content[0].text)
    # assert not result.isError and len(tree) >= 1
    # print("TODO 2 self-check passed.")

**Takeaway (the aha).** Look at what the client printed: a tool **name**, a human-readable **description**, and a typed **inputSchema** — and then the **call result**. The client **never imported `get_category_tree`**. It discovered the tool, read its self-describing contract, and invoked it **entirely over the protocol**. That description came straight from your docstring; that schema from your type hints. **Any** MCP-speaking agent could do exactly this — that's the standardization MCP buys you.

## 5. Expose the RAG and the category schema as tools

One tool is a demo. An agent needs a **toolbox**. We add the two operations the TD5 agent leans on most:

- **`search_products`** — semantic search over the catalog. **This is your TD3 RAG, behind a tool interface.** The agent will call it to find similar existing products before writing a new entry.
- **`get_category_attributes`** — given a leaf category, return its applicable attribute schema. This is a **structured lookup**, deliberately *not* RAG (a contract should be fetched exactly, not by semantic search).

(The write op `create_product` and `get_product` live in the mini-project — see §6 — to keep the notebook lean.)

In [ ]:
# TODO 3 -- expose the TD3 RAG as the MCP tool `search_products`.
# Register an MCP tool with this exact signature:
#     search_products(query: str, k: int = 3) -> list
#   1. decorate it with @mcp_server.tool();
#   2. in the body, call the pre-built retrieve(query, k=k) and return its list of hit dicts
#      (sku / name / category / price / short_description) -- a compact, JSON-serialisable list;
#   3. write the docstring FOR THE MODEL, e.g. "Semantic search over the product catalog; returns
#      up to k products most similar to the query." -- it's what the agent reads to decide to call it.
# This is the one place RAG meets MCP: the retriever you built in TD3 is now a discoverable tool.
# Note: when a tool returns a LIST, MCP sends ONE content block per item, so the client parses with
#       [json.loads(c.text) for c in res.content] (one json.loads per block).

# TODO: define and decorate search_products here
raise NotImplementedError

# Self-check: call it over the protocol for an audio query; expect >= 1 hit.
async with connect(mcp_server._mcp_server) as session:
    res = await session.call_tool("search_products", {"query": "noise cancelling headphones", "k": 3})
    hits = [json.loads(c.text) for c in res.content]
    assert not res.isError and len(hits) >= 1
    print("TODO 3 self-check passed. Top hits for 'noise cancelling headphones':")
    for h in hits:
        print(f"  - {h['name']}  ({h['category']})")

In [ ]:
# TODO 4 -- expose the category attribute schema as `get_category_attributes`.
# Register an MCP tool with this exact signature:
#     get_category_attributes(category: str) -> dict
#   1. decorate it with @mcp_server.tool();
#   2. look up the given LEAF category in `taxonomy` and return a dict describing its applicable
#      attributes. A simple, useful shape: {attribute_name: type_or_values_string, ...}.
#      The leaves live at taxonomy['categories'][i]['subcategories'][j], each with a
#      'category_attributes' list of {name, type, (values?), (unit?)} entries.
#   3. if the category is unknown, return an empty dict (don't crash) -- a tool should fail gracefully;
#   4. write a model-facing docstring, e.g. "Return the applicable attribute schema for a leaf category."
# This is a STRUCTURED lookup (a contract), deliberately NOT RAG.

# TODO: define and decorate get_category_attributes here
raise NotImplementedError

# Self-check: a known leaf returns its expected attribute keys.
async with connect(mcp_server._mcp_server) as session:
    res = await session.call_tool("get_category_attributes", {"category": "Headphones"})
    attrs = json.loads(res.content[0].text)
    assert not res.isError
    assert {"connectivity", "noise_cancellation"} <= set(attrs.keys()), attrs
    print("TODO 4 self-check passed. Headphones attributes:", list(attrs.keys()))

**Takeaway.** Your server now exposes a real **toolbox** — semantic search (your **TD3 RAG**, behind `search_products`) and the **category schema** (`get_category_tree`, `get_category_attributes`) — every one **discoverable and callable over the protocol**, with schemas and descriptions generated straight from your type hints and docstrings. There is **no agent here yet**, and that's deliberate: TD4 builds the toolbox; **TD5** builds the agent. There you'll write the **reason → act → observe** loop that connects to *this exact server*, discovers these tools, and lets the model decide — on its own — which to call and with what arguments, turning a one-shot tool call into a working agent.

## A short reflection

**Question — MCP vs. just passing `tools=` to the API.** An MCP client can translate a discovered tool catalog into the model's native `tools=` format (you'll do exactly that in **TD5**). So you *could* skip MCP and hand the model a **hand-written** `tools=` list directly. What does putting your operations behind an MCP **server** actually buy you — and what does it **cost**? Name **one** situation where the ad-hoc `tools=` approach is perfectly fine, and **one** where the protocol clearly wins.

**Your answer** — write it yourself. You may use AI to help you think it through, but the written answer must be in your own words.

_..._

## 6. Wrap-up + your mini-project

What you take away from TD4 — the transferable skill:

- **Expose capabilities as MCP tools.** Decorate a function with `@mcp_server.tool()`; the model reads your **docstring + type hints** as the contract. A client **discovers and calls** your tools over a standard protocol, **without importing your code** — that's the standardization + discoverability MCP gives you.
- **Structured lookups go through MCP, generation grounds in RAG.** `get_category_tree` / `get_category_attributes` are exact contracts (MCP); `search_products` *is* your TD3 RAG, now a tool.

**Where this returns:** in **TD3** you built `retrieve`; here it became the tool **`search_products`**; in **TD5** the agent connects to this server, discovers these tools, and orchestrates them to enrich a catalog from a supplier email.

---

### Mini-project brief — a standalone PIM MCP server *(build it yourself)*

*(Markdown only — no code here. You build this **from scratch** with Claude Code; we ship no scaffold. The whole point is that you can.)*

The notebook ran the server **in-memory** (in-process), which is clean for teaching. But MCP's real value is **out-of-process, language-agnostic** access — a server another program (your TD5 agent, Claude Desktop) can **spawn** and talk to over a transport. So the mini-project uses **stdio** transport.

### Where to build it
Create a **`mini_project/`** folder next to this notebook (`notebooks/TD4_mcp/mini_project/`) and put **all** the mini-project code there.

### What to build
1. **`pim_server.py`** — a standalone **stdio** MCP server (`FastMCP`, run with `mcp.run()` over stdio) exposing **five** PIM tools. The **ChromaDB index is the source of truth** for products — build it once from `products.csv` into a **persistent** store (`chromadb.PersistentClient`, exactly as in §1), then read **and write** it through the tools:
   - `search_products(query)` — semantic search over the persistent ChromaDB index (wraps your TD3 RAG);
   - `get_product(sku)` — **read** one product back **from ChromaDB** by its id (e.g. `collection.get(ids=[sku])`), returning its document + metadata;
   - `get_category_tree()` — over `taxonomy.json`;
   - `get_category_attributes(category)` — over `taxonomy.json`;
   - `create_product(...)` — the **write** operation TD5 will call: embed the new product with the **same MiniLM** model and `collection.add(...)` it **into ChromaDB**. Because it's the same vector space, the product is **immediately searchable** by `search_products` — the TD3 freshness aha, now exposed as a tool.
2. **the demo runs from Claude Desktop.** Register your server in Claude Desktop's MCP config (`claude_desktop_config.json` — an entry under `mcpServers` giving the command + args to launch `pim_server.py`, e.g. your venv's `python` and the absolute path), restart Claude Desktop, and confirm the tools appear. Then **ask Claude Desktop questions about your catalog** — "what noise-cancelling headphones do we carry under €300?", "add this product and then find it" — and watch it discover and call your tools. *This* is the payoff: a client you didn't write, driven in natural language, using your PIM server. (A tiny stdio `client_demo.py` that calls each tool is a fine extra sanity check, but Claude Desktop is the demo.)
3. hygiene: **no API key in the code** (Claude Desktop brings its own model — your server needs none), a **`requirements.txt`**, and a short **`README.md`** with run instructions: how to build the index, the exact `claude_desktop_config.json` snippet, and how to ask the questions.

Keep it **minimal** — a working server, not a product. It must be **spawnable by Claude Desktop now and by your TD5 agent later**.

### 🚀 Going further (optional — no solutions; use Claude Code)
- After a `create_product` from Claude Desktop, ask it to `search_products` for the new item in the same conversation — confirm it's found seconds later (freshness, end to end, with no reindexing).
- Write the tiny stdio `client_demo.py` too (`mcp.client.stdio`) and call each tool from Python — feel the out-of-process boundary the notebook's in-memory transport hid.
- Add a `delete_product(sku)` tool and think about what other write operations a real PIM agent would need. ✅